# Análisis exploratorio de la demanda turística rural

Este notebook analiza la tabla `gold_tourism_demand_monthly.parquet`.

**Objetivos:**

- comprobar la estructura y cobertura del dataset;
- analizar la evolución temporal de viajeros y pernoctaciones;
- estudiar la estacionalidad;
- comparar provincias;
- revisar los indicadores de ocupación y presión turística;
- analizar el caso de Santa Cruz de Tenerife;
- dejar conclusiones reproducibles para las siguientes fases del TFM.

La unidad de análisis es una combinación de **provincia y mes**.

## 1. Configuración y carga de datos

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)
pd.set_option("display.float_format", lambda value: f"{value:,.2f}")

PROJECT_ROOT = Path.cwd()

# Permite ejecutar el notebook tanto desde la raíz como desde notebooks/.
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

GOLD_PATH = (
    PROJECT_ROOT
    / "data"
    / "gold"
    / "gold_tourism_demand_monthly.parquet"
)

if not GOLD_PATH.exists():
    raise FileNotFoundError(
        f"No se encontró el dataset gold en: {GOLD_PATH}"
    )

df = pd.read_parquet(GOLD_PATH)
df["date_month"] = pd.to_datetime(df["date_month"])

print(f"Ruta cargada: {GOLD_PATH}")
print(f"Filas: {len(df):,}")
print(f"Columnas: {len(df.columns)}")

## 2. Comprobaciones iniciales

In [ ]:
summary = pd.Series(
    {
        "filas": len(df),
        "columnas": len(df.columns),
        "territorios": df["territory_id"].nunique(),
        "primer_mes": df["date_month"].min(),
        "último_mes": df["date_month"].max(),
        "claves_duplicadas": df.duplicated(
            ["territory_id", "month_id"]
        ).sum(),
        "filas_provisionales": df["is_provisional"]
        .fillna(False)
        .sum(),
    }
)

summary

In [ ]:
df.head()

In [ ]:
df.dtypes.to_frame("tipo")

### Interpretación esperada

El dataset debe contener 50 provincias, cero claves duplicadas y una cobertura temporal desde enero de 2005 hasta mayo de 2026. Los datos desde junio de 2025 están marcados como provisionales.

## 3. Valores nulos en variables principales

In [ ]:
main_columns = [
    "travellers_total",
    "overnight_stays_total",
    "average_stay",
    "establishments_estimated",
    "places_estimated",
    "occupancy_rate_pct",
    "weekend_occupancy_rate_pct",
    "room_occupancy_rate_pct",
    "staff_employed",
    "overnight_stays_yoy_change_pct",
    "tourism_pressure_index",
]

missingness = (
    df[main_columns]
    .isna()
    .agg(["sum", "mean"])
    .T
    .rename(
        columns={
            "sum": "nulos",
            "mean": "proporción_nulos",
        }
    )
)

missingness["porcentaje_nulos"] = (
    missingness["proporción_nulos"] * 100
)

missingness[
    ["nulos", "porcentaje_nulos"]
].sort_values("porcentaje_nulos", ascending=False)

Los nulos no se imputan en esta fase. Deben interpretarse como valores no publicados o no disponibles en la fuente original.

## 4. Estadísticos descriptivos

In [ ]:
descriptive_columns = [
    "travellers_total",
    "overnight_stays_total",
    "average_stay",
    "establishments_estimated",
    "places_estimated",
    "occupancy_rate_pct",
    "weekend_occupancy_rate_pct",
    "room_occupancy_rate_pct",
    "staff_employed",
    "tourism_pressure_index",
]

df[descriptive_columns].describe().T

## 5. Evolución mensual agregada de España

La suma provincial permite observar la evolución general de la demanda turística rural. No debe mezclarse con agregados autonómicos o nacionales procedentes de otras tablas.

In [ ]:
national_monthly = (
    df.groupby("date_month", as_index=False)
    .agg(
        travellers_total=("travellers_total", "sum"),
        overnight_stays_total=("overnight_stays_total", "sum"),
    )
    .sort_values("date_month")
)

national_monthly.tail()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(national_monthly["date_month"], national_monthly["overnight_stays_total"])
ax.set_title("Evolución mensual de las pernoctaciones rurales")
ax.set_xlabel("Mes")
ax.set_ylabel("Pernoctaciones")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(national_monthly["date_month"], national_monthly["travellers_total"])
ax.set_title("Evolución mensual de los viajeros en turismo rural")
ax.set_xlabel("Mes")
ax.set_ylabel("Viajeros")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Comparación anual

Para evitar comparar un año incompleto con años completos, se muestran por separado los años cerrados y el año 2026.

In [ ]:
annual_summary = (
    df.groupby("year", as_index=False)
    .agg(
        travellers_total=("travellers_total", "sum"),
        overnight_stays_total=("overnight_stays_total", "sum"),
        months_available=("month_id", "nunique"),
        contains_provisional_data=("is_provisional", "max"),
    )
    .sort_values("year")
)

annual_summary.tail(10)

In [ ]:
complete_years = annual_summary[
    (annual_summary["months_available"] == 12)
    & ~annual_summary["contains_provisional_data"].fillna(False)
]

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(complete_years["year"], complete_years["overnight_stays_total"])
ax.set_title("Pernoctaciones rurales por año completo no provisional")
ax.set_xlabel("Año")
ax.set_ylabel("Pernoctaciones")
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

## 7. Efecto del periodo COVID-19

In [ ]:
covid_comparison = (
    df.assign(
        period_group=df["covid_period"].map(
            {True: "periodo_covid", False: "fuera_periodo_covid"}
        )
    )
    .groupby("period_group", as_index=False)
    .agg(
        monthly_mean_travellers=("travellers_total", "mean"),
        monthly_mean_overnight_stays=("overnight_stays_total", "mean"),
        observations=("territory_id", "size"),
    )
)

covid_comparison

Los meses abril de 2020, mayo de 2020 y noviembre de 2020 no aparecen en la tabla gold porque no contienen métricas provinciales de demanda publicadas. No deben interpretarse como demanda igual a cero.

## 8. Estacionalidad mensual

In [ ]:
seasonal_monthly = (
    df[
        ~df["covid_period"].fillna(False)
        & ~df["is_provisional"].fillna(False)
    ]
    .groupby(["month", "month_name"], as_index=False)
    .agg(
        mean_overnight_stays=("overnight_stays_total", "mean"),
        mean_occupancy_rate=("occupancy_rate_pct", "mean"),
        mean_pressure_index=("tourism_pressure_index", "mean"),
    )
    .sort_values("month")
)

seasonal_monthly

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(seasonal_monthly["month_name"], seasonal_monthly["mean_overnight_stays"])
ax.set_title("Pernoctaciones medias por mes del año")
ax.set_xlabel("Mes")
ax.set_ylabel("Pernoctaciones medias por provincia")
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(
    seasonal_monthly["month_name"],
    seasonal_monthly["mean_occupancy_rate"],
    marker="o",
)
ax.set_title("Ocupación media por plazas según mes")
ax.set_xlabel("Mes")
ax.set_ylabel("Ocupación media (%)")
ax.tick_params(axis="x", rotation=45)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 9. Provincias con mayor demanda reciente

Se utiliza el último año completo no provisional disponible: 2024.

In [ ]:
ranking_2024 = (
    df[df["year"] == 2024]
    .groupby(
        ["territory_id", "territory_name", "autonomous_community_name"],
        as_index=False,
    )
    .agg(
        travellers_total=("travellers_total", "sum"),
        overnight_stays_total=("overnight_stays_total", "sum"),
        average_occupancy_rate=("occupancy_rate_pct", "mean"),
        average_pressure_index=("tourism_pressure_index", "mean"),
    )
    .sort_values("overnight_stays_total", ascending=False)
)

ranking_2024.head(15)

In [ ]:
top_10 = ranking_2024.head(10).sort_values("overnight_stays_total")
fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(top_10["territory_name"], top_10["overnight_stays_total"])
ax.set_title("Provincias con más pernoctaciones rurales en 2024")
ax.set_xlabel("Pernoctaciones")
ax.set_ylabel("Provincia")
plt.tight_layout()
plt.show()

## 10. Demanda nacional y extranjera

In [ ]:
origin_summary = pd.Series(
    {
        "viajeros_residentes_españa": df["travellers_domestic"].sum(),
        "viajeros_extranjeros": df["travellers_foreign"].sum(),
        "pernoctaciones_residentes_españa": df["overnight_stays_domestic"].sum(),
        "pernoctaciones_extranjeros": df["overnight_stays_foreign"].sum(),
    }
)

origin_summary

In [ ]:
recent_origin = (
    df[df["year"] == 2024]
    .groupby("autonomous_community_name", as_index=False)
    .agg(
        travellers_domestic=("travellers_domestic", "sum"),
        travellers_foreign=("travellers_foreign", "sum"),
    )
)

recent_origin["foreign_share"] = (
    recent_origin["travellers_foreign"]
    / (recent_origin["travellers_domestic"] + recent_origin["travellers_foreign"])
)

recent_origin.sort_values("foreign_share", ascending=False).head(10)

## 11. Relación entre ocupación y presión turística

In [ ]:
pressure_sample = df[
    ["territory_name", "month_id", "occupancy_rate_pct", "tourism_pressure_index"]
].dropna()

pressure_sample[["occupancy_rate_pct", "tourism_pressure_index"]].corr()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(
    pressure_sample["occupancy_rate_pct"],
    pressure_sample["tourism_pressure_index"],
    alpha=0.25,
)
ax.set_title("Relación entre ocupación e índice de presión turística")
ax.set_xlabel("Ocupación por plazas (%)")
ax.set_ylabel("Índice de presión turística")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

El índice de presión turística incorpora la ocupación, pero también las pernoctaciones por plaza, la tendencia interanual y la ocupación de fin de semana. Por eso no debe esperarse una correlación perfecta.

## 12. Caso de estudio: Santa Cruz de Tenerife

In [ ]:
tenerife = (
    df[df["source_territory_code"] == "38"]
    .sort_values("date_month")
    .copy()
)

tenerife[
    [
        "territory_name",
        "month_id",
        "travellers_total",
        "overnight_stays_total",
        "average_stay",
        "occupancy_rate_pct",
        "tourism_pressure_index",
    ]
].tail(24)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(tenerife["date_month"], tenerife["overnight_stays_total"])
ax.set_title("Pernoctaciones rurales en Santa Cruz de Tenerife")
ax.set_xlabel("Mes")
ax.set_ylabel("Pernoctaciones")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
tenerife_recent = tenerife[tenerife["date_month"] >= "2022-01-01"]
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(tenerife_recent["date_month"], tenerife_recent["occupancy_rate_pct"], label="Ocupación")
ax.plot(tenerife_recent["date_month"], tenerife_recent["tourism_pressure_index"], label="Presión turística")
ax.set_title("Ocupación y presión turística en Santa Cruz de Tenerife")
ax.set_xlabel("Mes")
ax.set_ylabel("Índice 0-100")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
tenerife_seasonality = (
    tenerife[
        ~tenerife["covid_period"].fillna(False)
        & ~tenerife["is_provisional"].fillna(False)
    ]
    .groupby(["month", "month_name"], as_index=False)
    .agg(
        mean_overnight_stays=("overnight_stays_total", "mean"),
        mean_average_stay=("average_stay", "mean"),
        mean_pressure_index=("tourism_pressure_index", "mean"),
    )
    .sort_values("month")
)

tenerife_seasonality

## 13. Crecimiento interanual reciente por provincia

In [ ]:
latest_month = df["date_month"].max()

latest_snapshot = (
    df[df["date_month"] == latest_month][
        [
            "territory_name",
            "autonomous_community_name",
            "overnight_stays_total",
            "overnight_stays_yoy_change_pct",
            "occupancy_rate_pct",
            "tourism_pressure_index",
            "is_provisional",
        ]
    ]
    .sort_values("overnight_stays_yoy_change_pct", ascending=False)
)

print(f"Último mes disponible: {latest_month:%Y-%m}")
latest_snapshot.head(15)

Los datos del último mes son provisionales. Esta clasificación debe mantenerse visible en cualquier dashboard o análisis.

## 14. Resumen automático de hallazgos

In [ ]:
best_2024 = ranking_2024.iloc[0]
highest_season = seasonal_monthly.loc[seasonal_monthly["mean_overnight_stays"].idxmax()]
tenerife_peak = tenerife.loc[tenerife["overnight_stays_total"].idxmax()]

findings = {
    "provincia_líder_2024": best_2024["territory_name"],
    "pernoctaciones_provincia_líder_2024": int(best_2024["overnight_stays_total"]),
    "mes_con_mayor_demanda_media": highest_season["month_name"],
    "máximo_histórico_tenerife_mes": tenerife_peak["month_id"],
    "máximo_histórico_tenerife_pernoctaciones": int(tenerife_peak["overnight_stays_total"]),
    "último_mes_disponible": latest_month.strftime("%Y-%m"),
    "último_mes_es_provisional": bool(latest_snapshot["is_provisional"].all()),
}

pd.Series(findings)

## 15. Conclusiones

Completar esta celda después de ejecutar todas las anteriores.

Aspectos que deben comentarse:

1. evolución general de la demanda rural;
2. impacto visible del periodo COVID-19;
3. meses con mayor estacionalidad;
4. provincias con mayor demanda;
5. diferencias entre demanda nacional y extranjera;
6. relación entre ocupación y presión turística;
7. comportamiento específico de Santa Cruz de Tenerife;
8. limitaciones derivadas de datos provisionales y valores no publicados.

**Conclusión provisional:** la estructura gold permite comparar territorios y meses de forma coherente, pero los indicadores deben interpretarse como señales agregadas de demanda y presión turística, no como estimaciones directas de facturación o rentabilidad de una microempresa concreta.